In [10]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

/Users/liny/nlp-group-17/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
import sys
!{sys.executable} -m pip install -U "transformers[torch]" "accelerate>=1.1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 7.1 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.4
    Uninstalling transformers-5.5.4:
      Successfully uninstalled transformers-5.5.4

[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [29]:
df = pd.read_csv("dataset/AllProductReviews.csv")
print(f"Raw Dataset: \n{df.head(1)}")

df = df.rename(columns={
    "ReviewBody": "reviewText",
    "ReviewStar": "overall"
})

# Keep only the needed column
df = df[["reviewText", "overall"]]

# Remove the missing values (if exists)
print("")
print(df[["reviewText", "overall"]].isnull().sum())
df = df.dropna(subset=["reviewText", "overall"])

print("")
print(f"Dataset: \n{df.head(1)}")

Raw Dataset: 
                             ReviewTitle  \
0  Honest review of an edm music lover\n   

                                          ReviewBody  ReviewStar  \
0  No doubt it has a great bass and to a great ex...           3   

            Product  
0  boAt Rockerz 255  

reviewText    0
overall       0
dtype: int64

Dataset: 
                                          reviewText  overall
0  No doubt it has a great bass and to a great ex...        3


In [30]:
def convert_rating_to_label(rating):  # Convert the rating to Negative, Neutral, and Positive
    if rating in [1, 2]:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

In [31]:
df["label"] = df["overall"].apply(convert_rating_to_label)
print(df.head(2))

                                          reviewText  overall  label
0  No doubt it has a great bass and to a great ex...        3      1
1  This  earphones are unreliable, i bought it be...        1      0


In [33]:
label_names = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label_counts = df["label"].value_counts().sort_index()
label_counts_named = label_counts.rename(index=label_names)

print(label_counts_named)

label
negative    3432
neutral     1503
positive    9402
Name: count, dtype: int64


In [34]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df["reviewText"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("X_train size:", len(X_train))
print("X_test size:", len(X_test))
print("X_val size:", len(X_val))

X_train size: 11469
X_test size: 1434
X_val size: 1434


In [35]:
# Converts a pandas into a normal Python list

train_set = Dataset.from_dict(
    {
        "reviewText": X_train.to_list(),
        "label": y_train.to_list()
    }
)

test_set = Dataset.from_dict(
    {
        "reviewText": X_test.to_list(),
        "label": y_test.to_list()
    }
)

val_set = Dataset.from_dict(
    {
        "reviewText": X_val.to_list(),
        "label": y_val.to_list()
    }
)

In [36]:
model = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model)


def tokenizer_function(text):
    return tokenizer(
        text["reviewText"],
        padding=True,
        truncation=True,
        max_length=128
    )

In [37]:
tokenized_train_set = train_set.map(tokenizer_function, batched=True)
tokenized_test_set = test_set.map(tokenizer_function, batched=True)
tokenized_val_set = val_set.map(tokenizer_function, batched=True)

print(tokenized_train_set[0])

Map: 100%|██████████| 1434/1434 [00:00<00:00, 17808.15 examples/s]

{'reviewText': 'Pros -Solid builtGood sound qualityFits well in the earBetter calling qualityCons -You get Static shocks sometimes\n', 'label': 2, 'input_ids': [101, 4013, 2015, 1011, 5024, 2328, 24146, 2614, 3737, 8873, 3215, 2092, 1999, 1996, 4540, 20915, 3334, 4214, 3737, 8663, 2015, 1011, 2017, 2131, 10763, 28215, 2823, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [38]:
tokenized_train_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

tokenized_test_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

tokenized_val_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# Exploring the tokenizer

In [39]:
sample_text = train_set["reviewText"][0]
sample_tokens = tokenizer(sample_text, padding=True, truncation=True, max_length=128)

print("Original text (first 200 chars):\n", sample_text[:200])
print("\nNumber of input_ids:", len(sample_tokens["input_ids"]))
print("First 20 token IDs:", sample_tokens["input_ids"][:20])
print("First 20 decoded tokens:", tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"][:20]))
print("Attention mask (first 30 values):", sample_tokens["attention_mask"][:30])

Original text (first 200 chars):
 Pros -Solid builtGood sound qualityFits well in the earBetter calling qualityCons -You get Static shocks sometimes


Number of input_ids: 28
First 20 token IDs: [101, 4013, 2015, 1011, 5024, 2328, 24146, 2614, 3737, 8873, 3215, 2092, 1999, 1996, 4540, 20915, 3334, 4214, 3737, 8663]
First 20 decoded tokens: ['[CLS]', 'pro', '##s', '-', 'solid', 'built', '##good', 'sound', 'quality', '##fi', '##ts', 'well', 'in', 'the', 'ear', '##bet', '##ter', 'calling', 'quality', '##con']
Attention mask (first 30 values): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


# Observe subword tokenization

In [40]:
test_words = ["Worked", "Gadget", "Disappoints"]

for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:20s} → {tokens}")

Worked               → ['worked']
Gadget               → ['ga', '##dget']
Disappoints          → ['di', '##sa', '##pp', '##oint', '##s']


# Evaluation and Test before fine-tuning

In [41]:
id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

baseline_model = BertForSequenceClassification.from_pretrained(
    model,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5464.16it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [48]:
results = []

def run_val_and_test(model, model_name, tokenized_val_set, tokenized_test_set, compute_metrics, output_dir):
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=output_dir,
            report_to="none",
            per_device_train_batch_size=16,
        ),
        compute_metrics=compute_metrics
    )

    # Validation
    val_results = trainer.evaluate(eval_dataset=tokenized_val_set)

    print(f"\nValidation metrics {model_name}")
    print("Accuracy:", val_results["eval_accuracy"])
    print("Precision:", val_results["eval_precision"])
    print("Recall:", val_results["eval_recall"])
    print("F1:", val_results["eval_f1"])

    # Test
    test_results = trainer.predict(tokenized_test_set)

    print(f"\nTest metrics {model_name}")
    print("Accuracy:", test_results.metrics["test_accuracy"])
    print("Precision:", test_results.metrics["test_precision"])
    print("Recall:", test_results.metrics["test_recall"])
    print("F1:", test_results.metrics["test_f1"])

    # Store the results
    results.append({
        "model": model_name,
        "val_accuracy": val_results["eval_accuracy"],
        "val_precision": val_results["eval_precision"],
        "val_recall": val_results["eval_recall"],
        "val_f1": val_results["eval_f1"],
        "test_accuracy": test_results.metrics["test_accuracy"],
        "test_precision": test_results.metrics["test_precision"],
        "test_recall": test_results.metrics["test_recall"],
        "test_f1": test_results.metrics["test_f1"]
    })

    return val_results, test_results


In [43]:
pre_val_results, pre_test_results = run_val_and_test(
    model=baseline_model,
    model_name="before_fine_tuning",
    tokenized_val_set=tokenized_val_set,
    tokenized_test_set=tokenized_test_set,
    compute_metrics=compute_metrics,
    output_dir="/Users/liny/nlp-group-17/output/bert_pre_results",
)

/Users/liny/nlp-group-17/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,1.157646,0,0.166667,0.427545,0.166667,0.139245



Validation metrics before_fine_tuning
Accuracy: 0.16666666666666666
Precision: 0.42754465126025265
Recall: 0.16666666666666666
F1: 0.13924527962322822


/Users/liny/nlp-group-17/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



Test metrics before_fine_tuning
Accuracy: 0.17782426778242677
Precision: 0.45434542665655353
Recall: 0.17782426778242677
F1: 0.1645144907064456


# Evaluation and Test Full Fine Tuning

In [45]:
full_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2208.19it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [47]:
full_trainer = Trainer(
    model=full_model,
    args=TrainingArguments(
        output_dir="/Users/liny/nlp-group-17/output/bert_full_finetune",
        report_to="none",
        logging_dir="/Users/liny/nlp-group-17/logs",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=5,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        logging_steps=1000
    ),
    train_dataset=tokenized_train_set,
    eval_dataset=tokenized_val_set,
    compute_metrics=compute_metrics
)

full_trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/Users/liny/nlp-group-17/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
full_val_results, full_test_results = run_val_and_test(
    model=full_trainer.model,
    model_name="full_fine_tuning",
    tokenized_val_set=tokenized_val_set,
    tokenized_test_set=tokenized_test_set,
    compute_metrics=compute_metrics,
    output_dir="/Users/liny/nlp-group-17/output/bert_full_results",
)

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)